In [1]:
from pathlib import Path

# Шляхи
DRIVE_DIR = Path(r"E:\PythonProject\CNN_HW")
DATA_DIR = DRIVE_DIR / "DATA/IMAGES/cats_and_dogs_filtered"
PROJECT_DIR = DRIVE_DIR / "DZ/cnn_learn_control/result"
CHECKPOINTS_DIR = PROJECT_DIR / "checkpoints"
LOG_DIR = PROJECT_DIR / "logs"
MODEL_DIR = PROJECT_DIR / "model"

In [2]:
# Створення папки для збереження моделі та додаткових файлів
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Параметри
SEED = 42
IMAGE_SIZE = (250, 250)
BATCH_SIZE = 32
# скільки епох уже пройдено
INITIAL_EPOCH = 0
# загальна цільова кількість епох
EPOCHS = 5

In [4]:
import tensorflow as tf
from tensorflow.keras import layers,models
from tensorflow.keras.optimizers import Adam

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.2),
    layers.RandomTranslation(0.1, 0.1),
])


In [5]:

model = models.Sequential([
 layers.Input(shape=(*IMAGE_SIZE, 3)),
 data_augmentation,
 layers.Rescaling(1./255),
 # Блок 1
 layers.Conv2D(32, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.Conv2D(32, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.MaxPooling2D(),
 layers.Dropout(0.2),
 # Блок 2
 layers.Conv2D(64, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.Conv2D(64, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.MaxPooling2D(),
 layers.Dropout(0.25),
 # Блок 3
 layers.Conv2D(128, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.Conv2D(128, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.MaxPooling2D(),
 layers.Dropout(0.3),
 # Блок 4
 layers.Conv2D(256, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.Conv2D(256, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.MaxPooling2D(),
 layers.Dropout(0.35),
 # Блок 5
 layers.Conv2D(512, 3, padding="same", use_bias=False),
 layers.BatchNormalization(),
 layers.Activation("relu"),
 layers.GlobalAveragePooling2D(),
 # Класифікатор
 layers.Dense(512, activation="relu"),
 layers.BatchNormalization(),
 layers.Dropout(0.5),
 layers.Dense(256, activation="relu"),
 layers.Dropout(0.3),
 layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [6]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    r"E:\PythonProject\CNN_HW\data\images\cats_and_dogs_filtered\train",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    seed=SEED,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    r"E:\PythonProject\CNN_HW\data\images\cats_and_dogs_filtered\validation",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False,
)
AUTOTUNE = tf.data.AUTOTUNE


Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.


In [7]:
from keras.src.callbacks import CSVLogger, TensorBoard
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

callbacks = [
    # сохраняет лучшую версию модели по val_loss
    ModelCheckpoint(MODEL_DIR/"best_model.keras", monitor="val_loss",
                    save_best_only=True, verbose=1),

    # останавливает обучение, если val_loss не улучшается 8 эпох подряд
    EarlyStopping(monitor="val_loss", patience=8,
                  restore_best_weights=True, verbose=1),

    # уменьшает learning rate вдвое, если 3 эпохи нет улучшения
    ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1),
    CSVLogger(CHECKPOINTS_DIR / "training_log.csv", append=True),
    TensorBoard(log_dir=LOG_DIR),
]

In [8]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    initial_epoch=INITIAL_EPOCH,
    epochs=EPOCHS,
    callbacks=callbacks,
)

E:\PythonProject\CNN_HW\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5290 - loss: 0.8856  
Epoch 1: val_loss improved from None to 0.69709, saving model to E:\PythonProject\CNN_HW\DZ\cnn_learn_control\result\model\best_model.keras

Epoch 1: finished saving model to E:\PythonProject\CNN_HW\DZ\cnn_learn_control\result\model\best_model.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 168s 3s/step - accuracy: 0.5290 - loss: 0.8856 - val_accuracy: 0.5090 - val_loss: 0.6971 - learning_rate: 0.0010
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5660 - loss: 0.7805  
Epoch 2: val_loss did not improve from 0.69709
63/63 ━━━━━━━━━━━━━━━━━━━━ 158s 2s/step - accuracy: 0.5660 - loss: 0.7805 - val_accuracy: 0.5000 - val_loss: 0.7598 - learning_rate: 0.0010
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5470 - loss: 0.8293  
Epoch 3: val_loss did not improve from 0.69709
63/63 ━━━━━━━━━━━━━━━━━━━━ 154s 2s/step - accuracy: 0.5470 - loss: 0.8293 - val_accuracy: 0.4770 - val_loss: 0.6984 - learni